# Uplift Modeling in Marketing — Notebook 4: Causal Forest and Uplift Trees (S5)

> Continuation of `03_Meta_Learners_EN.ipynb`. This notebook does not share the previous kernel; it reloads the training/validation splits below. Notebook 03 and Section 4 remain frozen: no cell there was changed in this round. The confirmatory test remains hidden: this notebook never touches it.
>
> **Objective of S5:** compare causal estimators and specialized uplift trees (Causal Forest, Uplift Tree, Uplift Random Forest) against the surviving S4 references (X-learner + shallow tree, S-learner + LightGBM vanilla) and the response-targeting baseline, using exactly the development protocol established in S4 — repeated stratified holdout inside `train_df`, with a single complementary evaluation on `val_df`. **S5 does not choose the final configuration** — that is done at the end of this section, with explicit review before freezing and opening the sealed test in S6.


## Contents

- [Setup — Retaking from S1-S4](#setup)
- [Section 5 — Causal Forest and Uplift Trees](#s5)
    - [5.1 Protocol and Pre-Registered Hypotheses](#s5-1)
    - [5.2 Implementation and Sanity Checks](#s5-2)
    - [5.3 Repeated Stratified Holdout — Main Comparison](#s5-3)
    - [5.4 Paired Differences Against Baseline and S4 References](#s5-4)
    - [5.5 Additional Evaluation in Fixed Holdout](#s5-5)
    - [5.6 Correlation between Rankings](#s5-6)
    - [5.7 S5 Summary](#s5-7)
    - [5.8 Pre-S6 Decision — Taken After Review of S5 and Before Sealed Test Opening](#s5-8)

---

In [ ]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.exceptions import ConvergenceWarning

# Path bootstrap: allows `from src...` from the notebooks directory.
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import SEED
from src.i18n import make_lang
from src.viz import apply_plot_style

np.random.seed(SEED)
pd.set_option('display.max_columns', 50)
pd.set_option('display.precision', 4)
warnings.filterwarnings('ignore', category=FutureWarning)
# CausalForestDML (model_y='auto') inclui um WeightedLassoCVWrapper (solver
# SAG) among the first-stage candidates does not converge within the
# default max_iter on samples of this size; it does not affect the result
# modelos candidatos do nuisance model), mas polui a saída.
warnings.filterwarnings('ignore', category=ConvergenceWarning)

lang = make_lang('en')
apply_plot_style()


In [ ]:
# Stack causal de S5 — verificado contra a API real instalada nesta rodada
# (not assumed from memory or documentation of another version).
import importlib.metadata as importlib_metadata

from causalml.inference.tree import UpliftRandomForestClassifier, UpliftTreeClassifier
from econml.dml import CausalForestDML
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeRegressor

print('Versões instaladas realmente usadas nesta rodada:')
print(f"  econml:   {importlib_metadata.version('econml')}")
print(f"  causalml: {importlib_metadata.version('causalml')}")


<a id="setup"></a>

## Setup — Retaking from S1-S4

In the normal flow, `get_train_val` loads the persisted training/validation manifests from `02_Baseline_Propensity_PT` (`train_index.parquet`, `validation_index.parquet`, `dataset_manifest.json`) and validates the SHA-256 fingerprint and `n_rows` of the current dataset against the manifest — without rewriting the sealed test index, which is only written by notebook 2. The treatment, primary outcome, and original set of covariates are exactly the same as in S3/S4: `treatment` pooled (`any email` vs. `No E-Mail`), outcome `visit`, and `FEATURE_COLS` from `src/config.py`. We do not separate Men's/Women's E-Mail again in this section — it would be another estimator.


In [ ]:
from src.config import FEATURE_COLS, POOLED_TREATMENT_COL, PRIMARY_OUTCOME
from src.data import add_pooled_treatment, load_hillstrom
from src.splits import get_train_val

df = load_hillstrom()
df_pooled = add_pooled_treatment(df)
train_df, val_df = get_train_val(df_pooled, persist_test=False)

labels = lang({'header': 'Tamanho das partições (retomado de S1-S4)'})
print(f"{labels['header']}:")
print(f"  Training:   {len(train_df):>6} rows | treated: {int(train_df[POOLED_TREATMENT_COL].sum()):>6}")
print(f"  Validation: {len(val_df):>6} rows | treated: {int(val_df[POOLED_TREATMENT_COL].sum()):>6}")
print(f"  Features:   {FEATURE_COLS}")


<a id='s5'></a>
<a id="s5"></a>

## Section 5 — Causal Forest and Uplift Trees

S4 did not produce a unique winner: X-learner + shallow tree and S-learner +
LightGBM vanilla form a top tier without stable separation under repeated
holdout, and neither of them consistently beats the response-targeting baseline (4.9.3).
This section tests whether any causal estimator (`CausalForestDML`, which
orthogonalizes via nuisance models — including an `outcome` model) or
specialized uplift tree (`UpliftTree`/`UpliftRF`, which constructs splits
directly oriented to differences in response between treatment and control)
can produce an incremental ranking that survives sample exchange.
S5 is not a competition to surpass the Qini AUC 0.0627 of the fixed holdout of S4.6;
it is a test of stability under resampling, in the same spirit as 4.9.


<a id="s5-1"></a>

### 5.1 Protocol and Pre-Registered Hypotheses

**Frozen References for S4, Mandatory for Comparison:** X-learner +
`DecisionTreeRegressor(max_depth=4)` (top tier, holdout fix 0.0627, mean
0.0250 in repeated holdout of 4.9.2); S-learner + LightGBM vanilla (top tier,
mean 0.0232); response-targeting baseline — `fit_propensity_baseline`, which
estimates P(Y=1|X,T=1) (response propensity, the model that marketing would
already train without notion of incremental effect), not P(T=1|X) (assignment
propensity, irrelevant here because the design is a RCT). The historical internal baseline label is preserved in tables for continuity with S4, while the public English reading uses response-targeting baseline.

**Main Protocol — Identical to 4.9:** repeated stratified holdout within
`train_df` (never `val_df` nor the sealed test), 15 repetitions, split
75% fit / 25% evaluation, stratified by (treatment pooled × `visit`), seeds
`1000+rep`, the same `fit_idx`/`eval_idx` reused between all candidates of
the same repetition — allows paired differences by split. Before interpreting
any new result, we verify if the three frozen benchmarks reproduce the
numbers of 4.9.2/4.9.3 (X+Tree≈0.0250, S+LightGBM≈0.0232, baseline≈0.0209) —
as splits, seeds, data, and implementations are the same, this reproduction
is expected to be exact, not approximate; a material discrepancy would signal
a regression of implementation, not a new result.

**Registered Hypotheses Before Running** — none with automatic superiority expectation:

**Causal Forest.** By estimating heterogeneity directly and using
orthogonalization/nuisance models, it may produce a more stable ranking than
some meta-learners, but there is no hypothesis that it necessarily surpasses
the S4 references.

**Uplift Tree.** By constructing splits directly aimed at separating effect
(not at predicting outcome), it may find useful segments that traditional
outcome models do not prioritize; on the other hand, a single tree may present
high variance — the same pattern of instability already seen in the T-learner
of 4.9.2.

**Uplift Random Forest.** The aggregation of trees can stabilize the ranking of the Uplift Tree, but S4 has already shown (4.6→4.8) that bagging does not guarantee an improvement in Qini — it's a hypothesis to test, not an expectation of superiority.

**Criterion of conviction, registered before seeing any results:** the most convincing result is the one that maintains relative performance under repeated holdout; a high Qini only in the fixed `val_df` is not enough — S4 has already shown exactly this pattern (T+Tree: 0.0615 in the fixed holdout, 0.0039 in the repeated holdout).


<a id="s5-2"></a>

### 5.2 Implementation and Sanity Checks

**Inspected Real API in this Round** (not assumed to be in memory): `econml`
0.16.0, `causalml` 0.15.5 — printed versions in the import cell above.

**`CausalForestDML`.** Two non-default parameters, both required by the study design: `discrete_treatment=True` (treatment pooled is binary {0,1}, not continuous) and `model_t=DummyClassifier(strategy='prior')`. The latter exists because this is an RCT — the treatment assignment mechanism does not depend on X. The default `model_t='auto'` expands, via inspection of the source code of `econml.dml.dml._make_first_stage_selector`/`get_selector`, to a `GridSearchCV` over `RandomForestClassifier`/`LogisticRegressionCV`: a nuisance model flexible that would learn spurious variation of P(T=1) in function of X under a mechanism that is, by design, X-invariant. A sklearn-compatible estimator passed directly (not `'auto'`/list/string) falls into the `FixedModelSelector` branch of the selector — used as is, without grid search on top; no hack was needed to force the known propensity (2/3) in the API. `model_y` stays at the default `'auto'`, consistent with the rest of the project, which always regresses the probability of the binary outcome (never classifies).

**`UpliftTreeClassifier`/`UpliftRandomForestClassifier`.** Both require `treatment` as group labels (string), not 0/1 — `control_name` explicitly identifies which label is the control. The two `predict()` have **different** semantics, confirmed by inspection of the installed `.pyx` source, not presumed: `UpliftTreeClassifier.predict(X)` returns probabilities P(Y=1) by group (a column by `classes_`, control included) — it is not uplift directly, needs the subtraction `P(Y=1|treated) − P(Y=1|control)`, locating the columns via `classes_.index(...)`, never by fixed position.
`UpliftRandomForestClassifier.predict(X, full_output=True)` already calculates `delta_{group}` explicitly — we use that named column, not the positional output of `full_output=False` (which is also the uplift, but unlabeled).

**Non-default hyperparameters, documented before any results**
(no grid search, no adjustment after seeing Qini): `discrete_treatment`
and `model_t` of `CausalForestDML` (above); `control_name` and `random_state` of
`UpliftTreeClassifier`/`UpliftRandomForestClassifier` (technical API requirement +
reproducibility). The rest remains in the library default —
`n_estimators=100`/`cv=2`/`honest=True` (Causal Forest), `max_depth=3` (Tree),
`n_estimators=10`/`max_depth=5` (RF), `evaluationFunction='KL'` (both).

**Preprocessing without leakage.** The three new candidates reuse
`encode_meta_learner_features`/`build_meta_learner_encoder` — the same
S4 infrastructure, already with the structural guarantee that the encoder is
adjusted only in the `fit_df` of each repetition, never before the split (see
`repeated_stratified_holdout` in `src/evaluation.py`).

**Smoke tests** (synthetic data with randomized treatment and known heterogeneous effect, and a small real sample in memory — never the sealed test) confirmed, for the three wrappers: correct length, finite values, non-constant score, correct sign (positive correlation with the true effect) and bit-by-bit reproducibility under the same `random_state` — see `tests/test_suite.py`.

**Computational cost.** In isolated smoke tests, with data of the real size of a `fit_df` of repeated holdout (~28,800 rows × 18 one-hot columns), a fit of `CausalForestDML` with default hyperparameters took ~207s — much more expensive than the other candidates (seconds), which motivated caution regarding the total expected time of cell 5.3 below. However, in the final complete execution of 5.3, the 15 repeated holdouts containing the six candidates totaled approximately 781.5s (~13 min) — well below what an ingenuous extrapolation from the isolated smoke test would suggest, showing relevant variation in cost between executions/conditions of the environment.
`UpliftTreeClassifier`/`UpliftRandomForestClassifier` are cheap (seconds).


In [ ]:
# Configuração explícita de cada família — registrada antes de qualquer
# resultado (nenhum grid search, nenhum ajuste posterior). Impressos a
# partir dos próprios argumentos passados ao construtor — nem
# `CausalForestDML` nem `UpliftTreeClassifier`/`UpliftRandomForestClassifier`
# expõem `.get_params()` (verificado: `AttributeError`), diferente da
# convenção usual de estimadores scikit-learn.
labels = lang({'header': 'Parâmetros não-default de cada família (o resto é default da biblioteca)'})
print(f"{labels['header']}:\n")
print('CausalForestDML: discrete_treatment=True, model_t=DummyClassifier(strategy=\'prior\'), random_state=%r' % SEED)
print('UpliftTreeClassifier: control_name=\'control\', random_state=%r' % SEED)
print('UpliftRandomForestClassifier: control_name=\'control\', random_state=%r' % SEED)


<a id="s5-3"></a>

### 5.3 Repeated Stratified Holdout — Main Comparison

Six candidates, the same 15 splits/seeds reused across all (within `train_df`, never `val_df` nor the sealed test): the two S4 frozen references, the response-targeting baseline, and the three specialized uplift/ effect estimators of this section.


In [ ]:
from src.evaluation import paired_deltas, repeated_holdout_summary, repeated_stratified_holdout

tree_factory = lambda: DecisionTreeRegressor(max_depth=4, random_state=SEED)

s5_candidates = {
    'X+Tree(depth=4)': ('meta', 'X', tree_factory, False),
    'S+LightGBM(vanilla)': ('meta', 'S', None, False),
    'Baseline (propensão)': ('propensity', None, None, False),
    'CausalForest': ('causal_forest', None, None, False),
    'UpliftTree': ('uplift_tree', None, None, False),
    'UpliftRF': ('uplift_rf', None, None, False),
}

t0 = time.time()
s5_results = repeated_stratified_holdout(train_df, POOLED_TREATMENT_COL, PRIMARY_OUTCOME, s5_candidates, n_reps=15)
elapsed = time.time() - t0
s5_summary = repeated_holdout_summary(s5_results)

labels = lang({'header': 'Repeated stratified holdout — 15 splits (só em train_df)'})
print(f"{labels['header']} — tempo total: {elapsed:.1f}s:")
print(s5_summary.round(4))


**Insights:**

The three frozen benchmarks exactly reproduced the numbers from 4.9.2/4.9.3 — X+Tree(depth=4) mean 0.0250, S+LightGBM(vanilla) mean 0.0232, baseline mean 0.0209 — as expected, since splits, seeds, data, and implementations are the same; no implementation regression detected.

**UpliftTree has the highest mean among the six candidates (0.0254), ahead of X+Tree (0.0250), and the highest win rate (40.0% of splits).** But its median (0.0201) falls below that of X+Tree (0.0259) — a more right-skewed distribution, with a higher ceiling (maximum 0.0537, the highest of the six) pulling the mean up, while the typical split lags behind X+Tree. UpliftRF stays close to the baseline in mean (0.0201 vs. 0.0209) and has the lowest standard deviation of all six (0.0096) — the numerically most stable candidate — but never is the best candidate in any of the 15 splits (win rate 0%), a profile of "consistent middle-of-the-pack" rather than occasional leader.

CausalForest has the lowest mean (0.0131) and a wide dispersion (standard deviation 0.0163), including at least one split with negative Qini AUC (-0.0189) — performance below a random ranking in that specific resample. None of the three specialized uplift/ effect estimators established a stable and unidirectional advantage over the frozen S4 references in this comparison.

<a id="s5-4"></a>

### 5.4 Paired Differences Against Baseline and S4 References

For each of the three specialized uplift effect estimators: Δ against the response-targeting baseline,
Δ against X+Tree(depth=4), Δ against S+LightGBM(vanilla) — same splits,
difference calculated split by split (not isolated aggregated means).


In [ ]:
new_candidates = ['CausalForest', 'UpliftTree', 'UpliftRF']

deltas_vs_baseline = paired_deltas(s5_results, baseline_candidate='Baseline (propensão)').loc[new_candidates]
deltas_vs_xtree = paired_deltas(s5_results, baseline_candidate='X+Tree(depth=4)').loc[new_candidates]
deltas_vs_slgbm = paired_deltas(s5_results, baseline_candidate='S+LightGBM(vanilla)').loc[new_candidates]

labels = lang({
    'h1': 'Δ = Qini(candidato) − Qini(baseline de propensão), por split',
    'h2': 'Δ = Qini(candidato) − Qini(X+Tree(depth=4)), por split',
    'h3': 'Δ = Qini(candidato) − Qini(S+LightGBM(vanilla)), por split',
})
print(f"{labels['h1']}:")
print(deltas_vs_baseline.round(4))
print(f"\n{labels['h2']}:")
print(deltas_vs_xtree.round(4))
print(f"\n{labels['h3']}:")
print(deltas_vs_slgbm.round(4))


**Insights:**

**Against the response-targeting baseline, UpliftTree shows the most consistent signal of the three** — it wins in 10 of 15 splits (66.67%), with a median positive delta (+0.0077); it only has a difference of 9 of 15 (60%) observed for S+LightGBM against the same baseline in 4.9.3. UpliftRF also surpasses the baseline in most splits (60.0%), with a median positive delta (+0.0038) despite a slightly negative mean delta (-0.0007) — the same pattern of asymmetry already seen in 5.3, a few bad splits pulling the mean down while the typical split favors UpliftRF. CausalForest loses to the baseline in most splits (it wins in only 26.67%), with negative mean and median deltas — the weakest of the three in this comparison.

Against X+Tree(depth=4), UpliftTree presents a nearly tied descriptive protocol in the repeated run: Δ mean +0.0004, Δ median +0.0001, and 8/15 positive deltas (53.33%). CausalForest and UpliftRF present lower proportions of positive deltas, 6/15 (40.0%) and 4/15 (26.67%), respectively — the same pattern of "no stable separation" already recorded between X+Tree and S+LightGBM in 4.9.2 (53%). Against S+LightGBM(vanilla), UpliftTree again stays close to a tie (46.67%), while UpliftRF and CausalForest lean towards the negative side (33.33% and 20.0%).

Summarizing the pattern: UpliftTree is the only one of the three specialized uplift/ effect estimators with a directly positive and reasonably consistent signal — specifically against the baseline —, without stable separation, in the repeated protocol, of the two top-tier references of S4. UpliftRF shows a similar pattern, but weaker. CausalForest does not show a positive signal in any of the paired comparisons.

<a id="s5-5"></a>

### 5.5 Additional Evaluation in Fixed Holdout

A single evaluation, `train_df` complete → `val_df`, with the settings already
frozen for the three specialized uplift/ effect estimators (no adjustments made
after seeing this result). Complementary to repeated holdout, not a substitute —
S4 has already shown why a single holdout can be misleading (T+Tree: 0.0615 in
fixed holdout, 0.0039 in repeated holdout). A spectacular result here and
mediocre in the repeated holdout above would be read as instability, not as
victory.


In [ ]:
from src.evaluation import evaluate_multiple_rankings
from src.learners import (
    build_meta_learner_encoder, encode_meta_learner_features, fit_causal_forest,
    fit_propensity_baseline, fit_single_meta_learner, fit_uplift_random_forest, fit_uplift_tree,
    predict_causal_forest_uplift, predict_propensity_score, predict_single_meta_learner,
    predict_uplift_random_forest_uplift, predict_uplift_tree_uplift,
)

encoder_s5 = build_meta_learner_encoder(train_df)
X_train_s5 = encode_meta_learner_features(train_df, encoder_s5)
X_val_s5 = encode_meta_learner_features(val_df, encoder_s5)
treatment_train_s5 = train_df[POOLED_TREATMENT_COL].to_numpy()
y_train_s5 = train_df[PRIMARY_OUTCOME].to_numpy(dtype=float)

t0 = time.time()
x_tree_model = fit_single_meta_learner('X', X_train_s5, treatment_train_s5, y_train_s5, base_learner_factory=tree_factory)
s_lgbm_model = fit_single_meta_learner('S', X_train_s5, treatment_train_s5, y_train_s5)
propensity_model_s5 = fit_propensity_baseline(train_df, POOLED_TREATMENT_COL, PRIMARY_OUTCOME)
cf_model_s5 = fit_causal_forest(X_train_s5, treatment_train_s5, y_train_s5)
ut_model_s5 = fit_uplift_tree(X_train_s5, treatment_train_s5, y_train_s5)
urf_model_s5 = fit_uplift_random_forest(X_train_s5, treatment_train_s5, y_train_s5)
elapsed = time.time() - t0

scores_val = {
    'X+Tree(depth=4)': predict_single_meta_learner('X', x_tree_model, X_val_s5),
    'S+LightGBM(vanilla)': predict_single_meta_learner('S', s_lgbm_model, X_val_s5),
    'Baseline (propensão)': predict_propensity_score(propensity_model_s5, val_df),
    'CausalForest': predict_causal_forest_uplift(cf_model_s5, X_val_s5),
    'UpliftTree': predict_uplift_tree_uplift(ut_model_s5, X_val_s5),
    'UpliftRF': predict_uplift_random_forest_uplift(urf_model_s5, X_val_s5),
}
fixed_holdout_summary = evaluate_multiple_rankings(val_df[PRIMARY_OUTCOME].values, scores_val, val_df[POOLED_TREATMENT_COL].values)

labels = lang({'header': 'Holdout fixo — train_df completo → val_df'})
print(f"{labels['header']} — tempo total de fit: {elapsed:.1f}s:")
print(fixed_holdout_summary.round(4))


**Insights:**

The three specialized uplift/ effect estimators score significantly lower in this single fixed holdout evaluation (CausalForest 0.0125; UpliftTree 0.0105; UpliftRF 0.0123) than their averages in repeated holdout (0.0131; 0.0254; 0.0201, respectively) — the most striking contrast is that of UpliftTree, whose average in repeated holdout was the highest among the six candidates (0.0254), but whose Qini in this single `val_df` is the lowest among the six.

**This is the pattern mirrored from what happened with T+Tree in 4.9** (there: strong in fixed holdout, weak in repeated holdout; here: weak in fixed holdout, strong in repeated holdout for UpliftTree) — reinforcing, on the opposite side, the same lesson already recorded in the pre-registered conviction criterion in 5.1: a single result from `val_df` is not sufficient evidence, and here it would have been actively misleading about UpliftTree if read in isolation, exactly as anticipated in the pre-registered hypothesis before running. By the same pre-registered rule ("a spectacular result here and mediocre in repeated holdout would be read as instability, not as victory"), the symmetric logic applies: a weak isolated result in `val_df`, contradicted by a relatively stronger signal in repeated holdout, should not be read as disqualifying. This fixed holdout number is reported as another data point, not as a basis for ordering the three specialized uplift/ effect estimators among themselves.

X+Tree, S+LightGBM, and the response-targeting baseline reproduce exactly their already known numbers from S4 (0.0627 / 0.0415 / 0.0395), confirming that the mechanism of fixed holdout remains unchanged.

<a id="s5-6"></a>

### 5.6 Correlation between Rankings

Correlation of Spearman between the scores of the six candidates in `val_df`
(the same scores as 5.5) — just to help interpret why models differ (similar vs. divergent rankings), not a new selection.


In [ ]:
scores_df = pd.DataFrame(scores_val)
rank_corr = scores_df.corr(method='spearman')

labels = lang({'header': 'Correlação de Spearman entre rankings (val_df)'})
print(f"{labels['header']}:")
print(rank_corr.round(3))


**Insights:**

X+Tree and S+LightGBM — the two references of S4 — correlate more strongly with each other (ρ=0.634) than any other non-identical pair, consistent with the two forming a top tier in S4. UpliftTree and UpliftRF also correlate strongly with each other (ρ=0.793) — expected, since UpliftRF is an ensemble via bagging of trees built with the same split criterion focused on uplift.

**UpliftTree and UpliftRF are the two candidates with the weakest correlation — even slightly negative — with the ranking of the response-targeting baseline** (ρ=-0.043 and ρ=-0.057, respectively): qualitatively the most different from a pure propensity ranking response among the six candidates, consistent with being native uplift estimators, which divide directly by heterogeneity of effect instead of by probability of outcome. This qualitative distinction does not imply, by itself, better performance — of the two, only UpliftTree showed a consistent paired advantage over the baseline in 5.4 — but helps to explain why the ranking profile of UpliftTree diverges from the baseline more than those of the S4 references.

CausalForest correlates more with S+LightGBM (ρ=0.521) than with any other candidate — both pass through some form of outcome/effect regression, which plausibly explains part of this shared ranking structure, despite CausalForest's weaker absolute performance.

<a id="s5-7"></a>

### 5.7 S5 Summary

**None of the three specialized uplift/ effect estimators established a stable advantage over the frozen references of S4.** X-learner + shallow tree and S-learner + vanilla LightGBM continue in the top tier at the end of this round — no evidence collected in S5 eliminates them, and neither of them consistently outperforms them.

**UpliftTree is the most interesting case in this section, but not a confirmed winner.** It had the highest mean (0.0254) and the highest win rate (40.0%) among the six candidates in the repeated holdout. UpliftTree presented the highest observed proportion of positive deltas against the response-targeting baseline in this protocol, 10 out of 15 resamples (66.67%, median delta +0.0077) — only one repetition above the 9 out of 15 (60%) observed for S+LightGBM against the same baseline in S4 — but failed to establish a stable separation, in the repeated protocol, from X+Tree (median delta +0.0004, 8 out of 15 splits, 53.33%) and had the worst result among the six in the single fixed holdout (0.0105), a protocol sensitivity pattern that reflects, in the opposite direction, the already seen with T+Tree in 4.9. Its median (0.0201), lower than its mean, suggests an asymmetric distribution — carry this caveat forward, not just the number of highest emphasis.

**UpliftRF presents the most numerically stable profile (lowest standard deviation, 0.0096) but was never the best candidate in any of the 15 splits.** It stays close to the baseline in mean (0.0201 vs. 0.0209) and outperforms it in 60.0% of the splits, a moderate signal, weaker than that of UpliftTree.

**CausalForest was the weakest candidate in this round in both protocols** — lowest mean in the repeated holdout (0.0131), without a positive paired advantage against any of the three references, and the second worst result in the fixed holdout. This does not invalidate the family — reflects the performance of this pre-registered and conservative configuration (`model_t` X-invariant, default hyperparameters) in this dataset, not a general evaluation of Causal Forests.

**Response to the central question of S5 — do specialized uplift/ effect estimators outperform the S4 references or the response-targeting baseline consistently?** No, with the data and configuration evaluated so far — with the caveat that UpliftTree shows a positive and non-trivial signal specifically against the response-targeting baseline, which is worth carrying as a candidate for review, not as a confirmed winner.

**Limitations.** The computational cost of `CausalForestDML` — ~207s per fit in isolated smoke test — motivated caution regarding the budget of this round; in the final complete execution of 5.3, the 15 repeated holdouts with the six candidates totaled ~781.5s (~13 min) altogether. This cost limited this round to the default pre-registered configuration — no alternative configuration, tuned or not, was tested within the budget of this round (Absolute Rule #6). The asymmetry of the distribution of UpliftTree (median below the mean) was not investigated further. As in S4, the 15 splits of the repeated holdout overlap partially — they are not independent samples; no formal test of significance was calculated here, and it would not be the basis for a decision even if calculated.

**State at the end of S5:** X+Tree(depth=4), S+LightGBM(vanilla) and the response-targeting baseline remain as references; UpliftTree enters as an additional candidate to review, without replacing any of the previous ones; UpliftRF and CausalForest did not show competitive evidence in this round. **No final configuration was chosen.** The selection between these candidates is left for explicit review before freezing the configuration and opening the sealed test in S6, following the protocol defined at the end of S4.

**Next:** review of the results of S5 before proceeding — S6 has not been initiated.

<a id="s5-8"></a>

### 5.8 Pre-S6 Decision — Taken After Review of S5 and Before Sealed Test Opening

This decision was made using exclusively development data (`train_df`/`val_df`). **The sealed test has not been opened yet.**

**S6 Primary Model:** `UpliftTreeClassifier`, exactly with the configuration used in S5 (`control_name='control'`, `random_state=SEED`, all other hyperparameters at the library's default).

**Pre-specified Comparators:**
1. X-learner + `DecisionTreeRegressor(max_depth=4)`
2. S-learner + LightGBM vanilla
3. Response-targeting baseline

**Not Carried Over to S6 Main Evaluation:** UpliftRandomForest, CausalForestDML. Excluding them does not mean that these families are inadequate in general — it means that the pre-registered configurations evaluated in S5 did not present sufficient competitive evidence to justify increasing the sealed test evaluation multiplicity.

**Justification for UpliftTree** (S5 repeated holdout numbers, 15 repetitions, within `train_df`):

- highest Qini mean among the six candidates: 0.0254;
- highest global win rate among the six: 40.0%;
- 10 of 15 positive deltas against the response-targeting baseline;
- median delta against the baseline: +0.0077;
- practical/tactical tie with X+Tree: average delta +0.0004, median delta +0.0001, 8 of 15 wins (53.33%);
- absence of stable separation against S+LightGBM (5.4);
- weak result in the fixed holdout (Qini 0.0105) — explicitly maintained as a sample/protocol sensitivity alert, not discarded or hidden.

**This choice should not be described as statistical superiority evidence.** It is a primary model selection based on the development protocol already privileged throughout the project (repeated holdout), plus a tiebreaker favoring the simplicity and interpretability of a specialized uplift tree.

This decision is registered before the sealed test opening — see `artifacts/s6/preregistration.json`.
